## Introducción

Este cuaderno implementa una **versión adimensional** del modelo de difusión inhomogénea para el cálculo de $G(t)$. Las variables se adimensionalizan con $L_\text{ref} = 100\,\text{nm}$ y $D_\text{ref} = 2\times10^9\,\text{nm}^2/\text{s}$:

$$z' = z/L_\text{ref}, \quad D' = D/D_\text{ref}, \quad t' = t\, D_\text{ref}/L_\text{ref}^2.$$

Los parámetros físicos se reescalan: $R = 10$ (adim.), $z_c = 0.1$, $\delta = 0.02$, $\lambda = 0.5 L_\text{ref}\,\text{nm}^{-1}$, $D_s/D_\text{ref} = 0.05$. El objetivo es facilitar la comparación con soluciones analíticas en variables reducidas y verificar la consistencia de las unidades del modelo dimensional.

Se usan 16 modos radiales y 16 axiales con una malla bizona sinh-adaptativa en $z'$, y se paraleliza con `joblib` (4 núcleos). Los autovalores se reescalan a unidades físicas al final para validar contra los resultados del cuaderno dimensional `TFM_V5.ipynb`.

In [1]:
import numpy as np
from scipy.special import jv, jn_zeros
from scipy.sparse.linalg import eigsh, ArpackNoConvergence
from scipy.sparse import diags, identity
from scipy.interpolate import interp1d, CubicSpline
from scipy.integrate import quad
from joblib import Parallel, delayed
import matplotlib.pyplot as plt

# =================================================================================
# 1. Escalas de Referencia y Parámetros Adimensionales
# =================================================================================
L_ref = 100.0           # Longitud de referencia en nm (escala característica del sistema)
D_ref = 2e9             # Difusividad de referencia en nm²/s (valor máximo de D(z))
tau_ref = L_ref**2/D_ref # Tiempo característico en segundos
n_modes_radial = 16  # Cantidad de modos de R(ρ)
m_max = 16           # Cantidad de modos de Z(z)

# =================================================================================
# 2. Parámetros Físicos Adimensionales
# =================================================================================
# Conversión a variables adimensionales
def adimensionalizar():
    global z_adim, D_adim, h_adim, alpha_adim, varkappa_adim
    
    # Parámetros originales en unidades físicas
    R_fisico = 1e3                # 1000 nm (1 µm)
    z_c_fisico = 10.0             # 10 nm
    delta_fisico = 2.0            # 2 nm
    lambda_fisico = 0.5           # 0.5 nm⁻¹
    L_fisico = 100.0              # 100 nm
    varkappa_fisico = 0.0         # nm/s
    
    # Conversión a adimensional
    R = R_fisico / L_ref
    z_c = z_c_fisico / L_ref
    delta = delta_fisico / L_ref
    lambda_param = lambda_fisico * L_ref
    L = L_fisico / L_ref
    varkappa_adim = varkappa_fisico * L_ref / D_ref
    
    return R, z_c, delta, lambda_param, L

R, z_c, delta, lambda_param, L = adimensionalizar()

# =================================================================================
# 3. Función de Difusión Adimensional
# =================================================================================
def D(z_adim_val):
    """Perfil de difusión adimensional: D' = D/D_ref"""
    z_fisico = z_adim_val * L_ref  # Convertimos a nm para mantener la física
    S = 0.5 * (1 + np.tanh((z_fisico - z_c*L_ref)/(delta*L_ref)))
    D_fisico = 0.05*D_ref + (D_ref - 0.05*D_ref)*((1-np.exp(-lambda_param*z_fisico/L_ref))*(1 - S) + S)
    return D_fisico / D_ref  # Adimensional

# =================================================================================
# 4. Generación de la Malla Adimensional
# =================================================================================
N_z = 1000
alpha_grid = 3.0

# Generación de la malla física en nm
j_vals1 = np.arange(int(N_z*0.5))
j_vals2 = np.arange(int(N_z*0.5))

z1 = z_c*L_ref * np.sinh(alpha_grid * j_vals1 / (N_z*0.5 - 1)) / np.sinh(alpha_grid) # [0,z_c]
z2 = z_c*L_ref + (L_ref - z_c*L_ref) * np.sinh(alpha_grid*j_vals2 / (N_z*0.5 - 1)) / np.sinh(alpha_grid) # [z_c,L]

z_fisico = np.concatenate((z1[:-2],z2))

# Conversión a coordenadas adimensionales
z_adim = z_fisico / L_ref
h_adim = np.diff(z_adim)
D_adim = D(z_adim)

# =================================================================================
# 5. Sistema Radial Adimensional
# =================================================================================
alpha_zeros = jn_zeros(0, 16)
alpha_adim = alpha_zeros/(R*L_ref)  # α' = α*L_ref

# =================================================================================
# 6. Sistema Axial Adimensional (¡Núcleo del Cambio!)
# =================================================================================
def build_system(n):
    try:
        alpha = alpha_adim[n]
        N = len(z_adim)
        main_diag = np.zeros(N)
        upper_diag = np.zeros(N-1)
        lower_diag = np.zeros(N-1)

        # Nodos internos
        j_internos = np.arange(1, N-1)
        j_prev = j_internos - 1
        j_next = j_internos + 1

        # Promedios armónicos adimensionales
        D_prev = 2 * D_adim[j_internos] * D_adim[j_prev] / (D_adim[j_internos] + D_adim[j_prev] + 1e-30)
        D_next = 2 * D_adim[j_internos] * D_adim[j_next] / (D_adim[j_internos] + D_adim[j_next] + 1e-30)

        main_diag[j_internos] = (D_prev/h_adim[j_prev]**2 + D_next/h_adim[j_internos]**2) + alpha**2 * D_adim[j_internos]
        upper_diag[j_internos] = -D_next/h_adim[j_internos]**2
        lower_diag[j_prev] = -D_prev/h_adim[j_prev]**2

        # Condiciones de frontera adimensionales
        main_diag[0] = 2*D_adim[0]/h_adim[0]**2 + alpha**2*D_adim[0] - 2*varkappa_adim/h_adim[0]
        upper_diag[0] = -2*D_adim[0]/h_adim[0]**2
        
        main_diag[-1] = D_adim[-1]/h_adim[-1]**2 + alpha**2*D_adim[-1]
        lower_diag[-1] = -D_adim[-1]/h_adim[-1]**2

        A = diags([main_diag, upper_diag, lower_diag], [0, 1, -1], format='csr')
        eigenvalues_adim, eigenvectors = eigsh(A, k=16, which='SM', tol=1e-6)
        
        # Re-escalar autovalores a unidades físicas
        eigenvalues = eigenvalues_adim * D_ref / L_ref**2
        return (n, (eigenvalues, eigenvectors))
    
    except ArpackNoConvergence:
        return (n, (np.zeros(16), np.zeros((N, 16))))

axial_results = Parallel(n_jobs=4, verbose=1)(delayed(build_system)(n) for n in range(n_modes_radial))
eigenvalues_ax = {}
eigenvectors_ax = {}
for n, (evals, evecs) in sorted(axial_results, key=lambda x: x[0]):
    eigenvalues_ax[n] = evals
    eigenvectors_ax[n] = evecs

# =================================================================================
# 8. Post-procesamiento: Normalización y Visualización
# =================================================================================

# --------------------------
# 8.1 Normalizar modos axiales
# --------------------------
for n in range(n_modes_radial):
    for m in range(m_max):
        # Integración en coordenadas adimensionales
        integrand = eigenvectors_ax[n][:, m]**2 / D_adim
        norm = np.sqrt(np.trapz(integrand, z_adim)) * np.sqrt(L_ref/D_ref)
        
        if norm > 1e-12:
            eigenvectors_ax[n][:, m] /= norm
        else:
            eigenvectors_ax[n][:, m] = 0.0

# --------------------------
# 8.2 Graficar espectro de autovalores
# --------------------------
plt.figure(figsize=(10,6))
for n in range(n_modes_radial):
    evals = eigenvalues_ax[n]  # Ya en unidades físicas (s⁻¹)
    plt.semilogy(evals, 'o-', label=f'n={n}')
plt.xlabel('Modo Axial', fontsize=12)
plt.ylabel(r'$\Lambda^2$ (s⁻¹)', fontsize=12)
plt.title('Espectro de Autovalores', fontsize=14)
plt.legend()
plt.grid(True, which='both', linestyle='--')
plt.savefig('figuras/espectro_autovalores.pdf', bbox_inches='tight')
plt.close()

# --------------------------
# 8.3 Graficar modos axiales
# --------------------------
plt.figure(figsize=(10,6))
z_physical = z_adim * L_ref  # Convertir a nm
for n in range(3):
    for m in range(3):
        plt.plot(z_physical, eigenvectors_ax[n][:, m], 
                label=f'n={n}, m={m}', linewidth=1.5)
plt.xlabel('z (nm)', fontsize=12)
plt.ylabel('Z(z)', fontsize=12)
plt.title('Modos Axiales Físicos', fontsize=14)
plt.legend()
plt.grid(True)
plt.savefig('figuras/modos_axiales_fisicos.pdf', bbox_inches='tight')
plt.close()

# =================================================================================
# 9. Término Temporal y Coeficientes A_{nm}
# =================================================================================

# --------------------------
# 9.1 Función temporal T(t)
# --------------------------
def T(t, Lambda_sq):
    return np.exp(-np.abs(Lambda_sq) * t)

# Graficar decaimiento temporal
t_phys = np.logspace(-12, -3, 100)  # Tiempo físico en segundos
plt.figure(figsize=(10,6))
for n in range(3):
    for m in range(3):
        Lambda_sq = eigenvalues_ax[n][m]
        plt.loglog(t_phys, T(t_phys, Lambda_sq), 
                  label=f'n={n}, m={m}', linewidth=1.5)
plt.xlabel('Tiempo (s)', fontsize=12)
plt.ylabel('T(t)', fontsize=12)
plt.title('Decaimiento Temporal', fontsize=14)
plt.legend()
plt.grid(True, which='both')
plt.savefig('figuras/decaimiento_temporal.pdf', bbox_inches='tight')
plt.close()

# --------------------------
# 9.2 Cálculo de coeficientes A_{nm}
# --------------------------
rho0_phys = 750  # nm
z0_phys = 10     # nm
rho0_adim = rho0_phys / L_ref
z0_adim = z0_phys / L_ref

# Funciones de interpolación en coordenadas adimensionales
Z_interps = {
    n: [CubicSpline(z_adim, eigenvectors_ax[n][:, m]) 
        for m in range(m_max)]
    for n in range(n_modes_radial)
}

def compute_A_nm(n):
    A = np.zeros(m_max)
    for m in range(m_max):
        # Integral en z adimensional
        z_min = z_adim[0]
        z_max = z_adim[-1]
        integrand = lambda z: Z_interps[n][m](z)**2 / D(z*L_ref)  # D en unidades físicas
        integral, _ = quad(integrand, z_min, z_max, limit=100)
        
        # Evaluación en posición física convertida a adimensional
        Z_val = Z_interps[n][m](z0_adim)
        J0 = jv(0, alpha_adim[n] * rho0_adim)
        
        # Factor de conversión por adimensionalización
        norm_factor = (L_ref**3/D_ref)  # Ajuste dimensional
        A[m] = (J0 * Z_val) / (2 * np.pi * rho0_phys * integral * norm_factor)
    
    return A

A_nm = Parallel(n_jobs=4)(delayed(compute_A_nm)(n) for n in range(n_modes_radial))

# =================================================================================
# 10. Densidad de Probabilidad y Función de Correlación
# =================================================================================

# --------------------------
# 10.1 Densidad P(ρ,z,t)
# --------------------------
def compute_P(rho_phys, z_phys, t_phys):
    rho = rho_phys / L_ref
    z = z_phys / L_ref
    P = np.zeros((len(z), len(rho)))
    
    for i, r in enumerate(rho):
        J0_vals = jv(0, alpha_adim * r)
        for j, z_val in enumerate(z):
            sum_val = 0.0
            for n in range(n_modes_radial):
                for m in range(m_max):
                    Z = Z_interps[n][m](z_val)
                    T_val = T(t_phys, eigenvalues_ax[n][m])
                    sum_val += A_nm[n][m] * J0_vals[n] * Z * T_val
            P[j,i] = np.abs(sum_val)
    
    return P * (L_ref**-3)  # Convertir a densidad física (nm⁻³)

# --------------------------
# 10.2 Función de Correlación G(t)
# --------------------------
def compute_G(t_phys, N=40):
    G = 0.0
    rho_phys = np.linspace(0.3*R*L_ref, 0.7*R*L_ref, N)
    z_phys = np.linspace(9, 11, N)
    
    for r in rho_phys:
        r_adim = r / L_ref
        J0 = jv(0, alpha_adim * r_adim)
        for z in z_phys:
            z_adim = z / L_ref
            f = (3*z**2 - r**2)/(r**2 + z**2)**2.5  # Término físico en nm⁻³
            sum_term = 0.0
            for n in range(n_modes_radial):
                for m in range(m_max):
                    Z = Z_interps[n][m](z_adim)
                    T_val = T(t_phys, eigenvalues_ax[n][m])
                    sum_term += A_nm[n][m] * J0[n] * Z * T_val
            G += f * sum_term
    
    dr = (0.7*R*L_ref - 0.3*R*L_ref)/N
    dz = 2.0/N
    return (5/(16*np.pi)) * G * dr * dz * (L_ref**3)  # Unidades físicas

# --------------------------
# 10.3 Ejecutar y guardar resultados
# --------------------------
t_values = np.logspace(-12, -6, 50)
G_values = Parallel(n_jobs=4)(delayed(compute_G)(t) for t in t_values)

np.savetxt("resultados/G_t.dat", 
          np.column_stack((t_values, G_values)),
          header="Tiempo(s) G(t) [nm-3]")

# Graficar
plt.figure(figsize=(10,6))
plt.loglog(t_values, G_values, 'o-', label='Simulación')
plt.xlabel('Tiempo (s)', fontsize=12)
plt.ylabel('G(t) [nm⁻³]', fontsize=12)
plt.title('Función de Correlación Nano-NMR', fontsize=14)
plt.grid(True, which='both')
plt.savefig('figuras/correlacion_nanoNMR.pdf', bbox_inches='tight')
plt.close()

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  16 out of  16 | elapsed:   36.8s finished


UnicodeEncodeError: 'charmap' codec can't encode character '\u207b' in position 20: character maps to <undefined>

In [ ]:
import numpy as np
from scipy.special import jv, jn_zeros
from scipy.sparse.linalg import eigsh, ArpackNoConvergence
from scipy.sparse import diags, identity
from scipy.interpolate import interp1d, CubicSpline
from scipy.integrate import quad
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

##############################################################################
# 1. Parámetros Físicos (en nanómetros)
##############################################################################
# Conversión: 1 m = 1e9 nm.
D0 = 2e-9 * 1e18     # 2e-9 m²/s -> 2e9 nm²/s
D_s = D0*0.05        # 5% de D0
R = 1e-6 * 1e9       # 1e-6 m -> 1000 nm
L = 100              # nm
z_c = 10/L             # 10 nm (altura crítica)
delta = 2/L            # 1 nm (ancho de transición)
lambda_param = 0.5*L   # en nm⁻¹
n_modes_radial = 16  # Cantidad de modos de R(ρ)
m_max = 16           # Cantidad de modos de Z(z)
N_z = 1000           # Número de nodos en z
alpha_grid = 3.0     # Aglomeramiento de nodos cerca de z=0
varkappa = 0*L/D_0         # nm/s

##############################################################################
# 2. Función de Difusión Global D(z) (z en nm)
##############################################################################
def D(z_val):
    """Perfil de difusión completo con z en nm.
       D(z) = D0 [exp(-λz)(1-S(z)) + S(z)] donde 
       S(z) = 0.5 [1 + tanh((z-z_c)/δ)].
    """
    S = 0.5 * (1 + np.tanh((z_val - z_c) / delta))
    # OLD (sin D_s): return D0 * ((1-np.exp(-lambda_param * z_val)) * (1 - S) + S)
    return D_s+(D0-D_s) * ((1-np.exp(-lambda_param * z_val)) * (1 - S) + S)

##############################################################################
# 3. Generación de la malla en z (nm)
##############################################################################
# j_vals = np.linspace(0, N_z, N_z)
j_vals1 = np.arange(int(N_z*0.5))
j_vals2 = np.arange(int(N_z*0.5))

z1 = z_c * np.sinh(alpha_grid * j_vals1 / (N_z*0.5 - 1)) / np.sinh(alpha_grid) # [0,z_c]
z2 = z_c + (L - z_c) * np.sinh(alpha_grid*j_vals2 / (N_z*0.5 - 1)) / np.sinh(alpha_grid) # [z_c,L]

z = np.concatenate((z1[:-2],z2))/L
# Perfil de difusión
D_values = D(z)/D0

plt.figure(figsize=(8,5))
plt.plot(z, D_values/D0, 'b-', lw=1.5)
plt.xlabel('z (nm)')
plt.ylabel('D(z)/D₀')
plt.title('Perfil de Difusión')
plt.grid(ls='--', alpha=0.7)
plt.savefig('figuras/Dz_profile.pdf', bbox_inches='tight')
plt.close()

##############################################################################
# 4. Solución Radial R(ρ)
##############################################################################
alpha_zeros = jn_zeros(0, n_modes_radial)
alpha_n = alpha_zeros / R   # R en nm
rho_plot = np.linspace(0.01 * R, 0.99* R, 80)
J0_table = jv(0, alpha_n[:, None] * rho_plot)

# Graficar modos radiales
plt.figure(figsize=(10,6))
for n in range(min(9, n_modes_radial)):
    plt.plot(rho_plot, J0_table[n], label=f"n={n+1}", linewidth=1.5)
plt.xlabel("ρ (nm)", fontsize=12)
plt.ylabel("Amplitud", fontsize=12)
plt.title("Modos Radiales de Bessel", fontsize=14)
plt.legend()
plt.grid(True)
plt.savefig("figuras/modos_radiales.pdf", bbox_inches="tight")
plt.close()

##############################################################################
# 5. Solución Axial Z(z)
##############################################################################
def build_system(n):
    """
    Construye la matriz simétrica A para el modo radial n.
    
    Args:
        n (int): Índice del modo radial.
        
    Returns:
        A (csr_matrix): Matriz simétrica tridiagonal.
    """
    try:
        alpha = alpha_n[n]
        h = np.diff(z)
        N = len(z)
        main_diag = np.zeros(N)
        upper_diag = np.zeros(N-1)
        lower_diag = np.zeros(N-1)
        # for j in range(1, N-1):
        #     h_prev = h[j-1]
        #     h_next = h[j]
        #     D_prev = 2 * D_values[j] * D_values[j-1] / (D_values[j] + D_values[j-1] + 1e-30)
        #     D_next = 2 * D_values[j] * D_values[j+1] / (D_values[j] + D_values[j+1] + 1e-30)
        #     h_avg = 0.5 * (h_prev + h_next)
        #     # main_diag[j] = D_prev / (h_prev * h_avg) + D_next / (h_next * h_avg) + alpha**2 * D_values[j]
        #     # upper_diag[j] = -D_next / (h_next * h_avg)
        #     # lower_diag[j-1] = -D_prev / (h_prev * h_avg)
        #     upper_diag[j] = - D_next / h_avg**2
        #     main_diag[j]  = (D_next + D_prev) / h_avg**2 + alpha**2*D_values[j]
        #     lower_diag[j-1] = - D_prev / h_avg**2

        # Índices internos (j=1 a N-2)
        # j_internos = np.arange(1, N-1)
        
        # # Cálculos vectorizados
        # D_prev = 2 * D_values[j_internos] * D_values[j_internos-1] / (D_values[j_internos] + D_values[j_internos-1] + 1e-30)
        # D_next = 2 * D_values[j_internos] * D_values[j_internos+1] / (D_values[j_internos] + D_values[j_internos+1] + 1e-30)
        # h_avg = 0.5 * (h[j_internos-1] + h[j_internos])
        
        # # Asignación a diagonales
        # upper_diag[j_internos] = -D_next / h_avg**2
        # lower_diag[j_internos-1] = -D_prev / h_avg**2
        # main_diag[j_internos] = (D_prev + D_next) / h_avg**2 + alpha**2 * D_values[j_internos]
        
        # # Condiciones de frontera:
        # # En z=0: superficie reflectante: Z0 = Z1
        # main_diag[0]  = D0 + varkappa*h[0]
        # upper_diag[0] = -D0
        # # En z=L: Neumann: Z(N-1) = Z(N-2)
        # main_diag[-1] = 1.0
        # lower_diag[-1] = -1.0

        # A = diags([main_diag, upper_diag, lower_diag], [0, 1, -1])
        # eigenvalues, eigenvectors = eigsh(A, k=m_max, which='SM', tol=1e-6, maxiter=15000)

        # ---------------------------------------------------------------
        # 1. Índices internos (j=1 a N-2)
        # ---------------------------------------------------------------
        j_internos = np.arange(1, N-1)  # j=1, 2, ..., N-2
        j_prev = j_internos - 1         # j-1 = 0, 1, ..., N-3
        j_next = j_internos + 1         # j+1 = 2, 3, ..., N-1
    
        # ---------------------------------------------------------------
        # 2. Cálculo de coeficientes (vectorizado)
        # ---------------------------------------------------------------
        # Promedios armónicos de D en interfases
        D_prev = 2 * D_values[j_internos] * D_values[j_prev] / (D_values[j_internos] + D_values[j_prev] + 1e-30)  # D_{j-1/2}
        D_next = 2 * D_values[j_internos] * D_values[j_next] / (D_values[j_internos] + D_values[j_next] + 1e-30)  # D_{j+1/2}
        
        # Espaciados promedio
        h_avg_prev = 0.5 * (h[j_prev] + h[j_internos-1])  # h_avg para D_{j-1/2}
        h_avg_next = 0.5 * (h[j_internos] + h[j_next-1])  # h_avg para D_{j+1/2}
    
        # Términos de la matriz (simétricos)
        main_diag[j_internos] = (D_prev/(h[j_prev] * h_avg_prev)) + (D_next/(h[j_internos] * h_avg_next)) + alpha**2 * D_values[j_internos]
        upper_diag[j_internos] = -D_next / (h[j_internos] * h_avg_next)  # A[j, j+1]
        lower_diag[j_prev] = -D_prev / (h[j_prev] * h_avg_prev)          # A[j, j-1]
    
        # ---------------------------------------------------------------
        # 3. Condiciones de frontera - Simétricas
        # ---------------------------------------------------------------
        # Robin en z=0 (DZ' + κZ = 0)
        h0 = h[0]
        D0 = D_values[0]
        main_diag[0] = (2 * D0) / (h0 ** 2) - (2 * varkappa) / h0 + alpha**2 * D0  # Diagonal principal
        upper_diag[0] = - (2 * D0) / (h0 ** 2)  # Término superior
        lower_diag[0] = upper_diag[0]  # Simetría: A[0,1] = A[1,0]
    
        # Neumann en z=L (Z' = 0)
        hL = h[-1]
        DL = D_values[-1]
        main_diag[-1] = DL / hL**2 + alpha**2 * DL  # Diagonal principal
        lower_diag[-1] = - DL / hL**2  # Término inferior
        upper_diag[-1] = lower_diag[-1]  # Simetría: A[-1,-2] = A[-2,-1]
    
        # ---------------------------------------------------------------
        # 4. Ensamblar matriz tridiagonal simétrica
        # ---------------------------------------------------------------
        A = diags(
            diagonals=[main_diag, upper_diag, lower_diag],
            offsets=[0, 1, -1],
            shape=(N, N),
            format='csr'
        ) + 1e-10 * identity(N)
        eigenvalues, eigenvectors = eigsh(A, k=m_max, which='SM', tol=1e-6, maxiter=15000)
        return (n, (eigenvalues, eigenvectors))
    except ArpackNoConvergence:
        print(f"Advertencia: Modo axial para n={n} no convergió. Usando ceros.")
        return (n, (np.zeros(m_max), np.zeros((N, m_max))))

axial_results = Parallel(n_jobs=4, verbose=1)(delayed(build_system)(n) for n in range(n_modes_radial))
eigenvalues_ax = {}
eigenvectors_ax = {}
for n, (evals, evecs) in sorted(axial_results, key=lambda x: x[0]):
    eigenvalues_ax[n] = evals
    eigenvectors_ax[n] = evecs

# Normalizar los modos axiales en z
for n in range(n_modes_radial):
    for m in range(m_max):
        norm = np.sqrt(np.trapz(eigenvectors_ax[n][:, m]**2, z))
        # norm = np.sqrt(np.trapz(eigenvectors_ax[n][:, m]**2*D(z), z))
        if norm != 0:
            eigenvectors_ax[n][:, m] /= norm
# Graficar espectro de autovalores
for n in range(n_modes_radial):
    evals = eigenvalues_ax[n]
    plt.semilogy(evals, 'o-', label=f'n={n}')
plt.xlabel('Modo Axial')
plt.ylabel('Autovalor (s⁻¹)')
plt.title('Espectro de Autovalores')
plt.legend()
plt.grid(ls='--', alpha=0.7)
plt.savefig('figuras/eigenvalues.pdf', bbox_inches='tight')
plt.close()

# Graficar algunos modos axiales
plt.figure(figsize=(10,6))
for n in range(3,6): # range(min(3, n_modes_radial)):
    for m in range(min(3, m_max)):
        plt.plot(z, eigenvectors_ax[n][:, m], label=f"n={n}, m={m}", linewidth=1.5)
plt.xlabel("z (nm)", fontsize=12)
plt.ylabel("Z(z)", fontsize=12)
plt.title("Modos Axiales", fontsize=14)
plt.legend()
plt.grid(True)
plt.savefig("figuras/modos_axiales.pdf", bbox_inches="tight")
plt.close()

##############################################################################
# 6. Término Temporal T(t)
##############################################################################
# Ahora, para cada eigenvalor (de la parte axial) usaremos el valor absoluto
# para que T(t)=exp(-|λ| t) decaiga.
def T(t, lam):
    return np.exp(-np.abs(lam) * t)

# Graficamos T(t) para los modos del sistema axial

plt.figure(figsize=(10,6))
t_plot = np.logspace(-12, -3, 100)
for n in range(min(3,n_modes_radial)):
    for m in range(min(3, m_max)):
        lam = eigenvalues_ax[n][m]
        plt.loglog(t_plot, T(t_plot, lam), label=f"n={n}, m={m}", linewidth=1.5)
plt.xlabel("Tiempo (s)", fontsize=12)
plt.ylabel("T(t)", fontsize=12)
plt.title("Decaimiento Temporal T(t)", fontsize=14)
plt.legend()
plt.grid(True, which="both")
plt.savefig("figuras/T_t.pdf", bbox_inches="tight")
plt.close()

##############################################################################
# 7. Cálculo de Coeficientes A_{nm}
##############################################################################
# Se asume que la condición inicial es:
#     P(ρ, z, 0) ∝ δ(ρ-ρ0) δ(z-z0) / (2πρ0)
# Con ρ0 = 750 nm y z0 = 10 nm.
rho0 = 750  # nm
z0 = 10     # nm

# Creamos funciones de interpolación para los modos axiales (resultado de la parte axial)
Z_interps = {}
for n in range(n_modes_radial):
    # Z_interps[n] = [interp1d(z, eigenvectors_ax[n][:, m], bounds_error=False, fill_value=0.0) for m in range(m_max)]
    Z_interps[n] = [CubicSpline(z, eigenvectors_ax[n][:, m], extrapolate=False) for m in range(m_max)] 

def compute_A_nm(n):
    # Calcula la norma en la dirección z para el modo m
    Z_norms = np.array([quad(lambda z_val: Z_interps[n][m](z_val)**2 / D(z_val), z[0], z[-1], limit=300)[0] for m in range(m_max)])
    # Evalúa cada modo en z = 15 nm
    Z_m_z0 = np.array([Z_interps[n][m](z0) for m in range(m_max)])
    # Evalúa el modo radial en ρ0 = 750 nm
    J0_rho0 = jv(0, alpha_n[n] * rho0)
    valid = (Z_norms > 1e-25) & (np.abs(Z_m_z0) > 1e-12)
    return (J0_rho0 * Z_m_z0 * valid) / (2 * np.pi * rho0 * (Z_norms + 1e-30))

A_nm = Parallel(n_jobs=4)(delayed(compute_A_nm)(n) for n in range(n_modes_radial))

##############################################################################
# 8. Densidad de Probabilidad P(ρ,z,t)
##############################################################################
def compute_P(rho_vals, z_vals, t):
    P = np.zeros((len(z_vals), len(rho_vals)))
    # Para cada modo se usa T_{n,m}(t)= exp(-|λ_{n,m}|t)
    for i, rho in enumerate(rho_vals):
        J0_vals = jv(0, alpha_n * rho)
        for j, z_val in enumerate(z_vals):
            s = 0.0
            for n in range(n_modes_radial):
                for m in range(m_max):
                    Z_val = Z_interps[n][m](z_val)
                    T_val = np.exp(np.clip(-np.abs(eigenvalues_ax[n][m]) * t, -700, 700))
                    s += A_nm[n][m] * J0_vals[n] * Z_val * T_val
            P[j, i] = np.abs(s)
    return P

rho_vals = np.linspace(0.01 * R, 0.99 * R, 50)
z_vals = np.linspace(0.01*L, 0.99*L, 50)
for t_val in [1e-12,1e-11,1e-10,1e-9,1e-8,1e-7,1e-6,1e-5,1e-4,1e-3]:
    P_val = compute_P(rho_vals, z_vals, t=t_val)
    
    plt.figure(figsize=(10,8))
    plt.contourf(rho_vals, z_vals, P_val, 50, cmap="plasma")
    plt.xlabel("ρ (nm)", fontsize=12)
    plt.ylabel("z (nm)", fontsize=12)
    plt.title(f"Densidad P(ρ,z,t) para t = {t_val}", fontsize=14)
    plt.colorbar()
    plt.savefig(f"figuras/P_contour_{t_val}.pdf", bbox_inches="tight")
    plt.close()

##############################################################################
# 9. Función de Correlación G(t)
##############################################################################
def compute_G(t, N_quad=40):
    # Dominios en nm: ρ en [0.3R, 0.7R] y z en [9,11] nm.
    z_min = 7
    z_max = 13
    j_nodes = np.linspace(z_min, z_max, N_quad)
    
    # Mallado no uniforme en z
    z_nodes = z_min + (z_max - z_min) * np.sinh(alpha_grid*j_nodes / (N_quad - 1))/np.sinh(alpha_grid) # [z_min,z_max]

    # calculamos pesos en z con la regla del trapecio no uniforme
    dz = np.diff(z_nodes)
    wz = np.empty_like(z_nodes)
    wz[0]    = dz[0]/2
    wz[-1]   = dz[-1]/2
    wz[1:-1] = 0.5*(dz[:-1]+dz[1:])

    # Mallado uniforme en rho
    rho_nodes = np.linspace(0.1 * R, 0.9 * R, N_quad)
    total = 0.0
    for rho in rho_nodes:
        J0_vals = jv(0, alpha_n * rho)
        for j,z_val in enumerate(z_nodes):
            f = (3 * z_val**2 - rho**2) / (rho**2 + z_val**2 + 1e-30)**2.5
            s = 0.0
            for n in range(n_modes_radial):
                for m in range(m_max):
                    Z_val = Z_interps[n][m](z_val)
                    T_val = np.exp(np.clip(-np.abs(eigenvalues_ax[n][m]) * t, -700, 700))
                    s += A_nm[n][m] * J0_vals[n] * Z_val * T_val
            total += np.abs(rho*f*s*wz[j])
    dr = (0.9 * R - 0.1 * R) / N_quad
    result = (5 / (16 * np.pi)) * (total * dr)
    return result

t_values = np.logspace(-12, -6, 50)
G_values = Parallel(n_jobs=4)(delayed(compute_G)(t) for t in t_values)

np.savetxt("data/G_vs_t.dat", np.c_[t_values, G_values], header="Tiempo(s) G(t)", fmt="%.3e")

plt.figure(figsize=(10,6))
plt.loglog(t_values, G_values, "o-", label="Simulación", color="navy", markersize=6, linewidth=1.5)
plt.plot(t_values, 1e-8 * t_values**-1.5, "--", label=r"Teoría $t^{-3/2}$", color="crimson", linewidth=1.5)
plt.xlabel("Tiempo (s)", fontsize=12)
plt.ylabel("G(t)", fontsize=12)
plt.title("Función de Correlación Nano-NMR", fontsize=14)
plt.legend()
plt.grid(which="both", alpha=0.4)
plt.savefig("figuras/correlacion_nanoNMR.pdf", bbox_inches="tight", dpi=300)
plt.close()

## Conclusiones

La formulación adimensional confirma la consistencia del modelo dimensional y sirve como banco de pruebas para extensiones futuras:

- Los autovalores adimensionales $\lambda'^2$ rescalados producen autovalores físicos $\lambda^2 = \lambda'^2 D_\text{ref}/L_\text{ref}^2$ que son compatibles con los obtenidos directamente en el cuaderno dimensional `TFM_V5.ipynb` (cuando se ajustan los parámetros de forma equivalente).
- La normalización de los modos axiales en coordenadas adimensionales con peso $1/D'(z')$ implementa correctamente la ortogonalidad del operador de Sturm-Liouville.
- El cuaderno sirve como prototipo exploratorio: utiliza 16 modos y una geometría distinta ($R = 1000\,\text{nm}$, $L = 100\,\text{nm}$), diferenciándose de la versión definitiva del TFM.

Los resultados definitivos del TFM se obtienen en el cuaderno dimensional `TFM_V5.ipynb` con mallas más finas ($N_z = 90\,000$) y mayor número de modos.